# 3회차 · 지도학습 2 및 모델 평가 — Colab 실습
**2026 경기도 공유학교 · 인공지능 기초 (김순찬 교수)** · 성취기준 [인일 03-04] 인공지능 모델의 성능을 비교하고 개선할 수 있다.

## 오늘의 학습 목표
1. 훈련 데이터와 테스트 데이터를 나누어 모델을 **공정하게** 평가한다.
2. 정확도·정밀도·재현율·F1·혼동행렬을 읽고, **정확도만 믿으면 안 되는 경우**를 설명한다.
3. 과적합을 눈으로 확인하고, **교차검증**과 **하이퍼파라미터 튜닝**으로 모델을 개선한다.
4. 여러 모델의 성능을 표로 비교해 **개선 방향을 근거와 함께 결정**한다.

## 사용법
- 셀을 **위에서부터 순서대로** 실행합니다. (셀 클릭 → `Shift`+`Enter`)
- 오류가 나면 **런타임 → 런타임 다시 시작** 후 처음부터 다시 실행합니다.
- 🧩 표시가 있는 셀은 여러분이 직접 채우는 셀입니다. 정답은 바로 아래 ▶ 정답 보기를 누르면 나옵니다.

## 0. 준비 — 도구 불러오기와 한글 폰트
파이썬에서 표를 다루는 `pandas`, 숫자 계산 `numpy`, 그림 `matplotlib`, 머신러닝 `scikit-learn`을 불러옵니다.
Colab에는 한글 폰트가 없어 그래프의 한글이 □□로 깨지므로 나눔 폰트를 설치합니다. (30초쯤 걸립니다. `try` 안의 설치가 실패해도 그래프는 그려집니다.)

In [ ]:
import matplotlib.pyplot as plt, matplotlib as mpl, numpy as np, pandas as pd
import subprocess, sys
try:
    subprocess.run(['apt-get','-qq','install','-y','fonts-nanum'], check=False, capture_output=True)   # Colab 한글 폰트
    import matplotlib.font_manager as fm; fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf'); plt.rcParams['font.family']='NanumGothic'
except Exception:
    plt.rcParams['font.family'] = 'Noto Sans CJK JP'   # 로컬 검증용
plt.rcParams['axes.unicode_minus'] = False
import warnings; warnings.filterwarnings('ignore')
print('준비 완료! numpy', np.__version__, '| pandas', pd.__version__)

## 1. 데이터 불러오기 — 와인 품종 분류 데이터
scikit-learn에 들어 있는 **와인 데이터**(178병)를 씁니다. 13가지 화학 성분(알코올 도수, 색깔 진하기 등)을 보고 **품종 3가지(0, 1, 2) 중 무엇인지** 맞히는 **분류** 문제입니다.
- `X` = 입력(13개 성분), `y` = 정답(품종 번호)
- 출력에서 **행 수(178)와 열 수(13)**, 그리고 품종별로 몇 병씩 있는지 확인하세요.

In [ ]:
from sklearn.datasets import load_wine
wine = load_wine(as_frame=True)          # as_frame=True → pandas 표로 받기
X = wine.data                             # 입력: 13개 성분
y = wine.target                           # 정답: 품종 0/1/2
print('데이터 크기 (행, 열):', X.shape)
print('품종별 개수:\n', y.value_counts().sort_index())
X.head()                                  # 앞 5줄 미리보기

## 2. 훈련 / 테스트 데이터 나누기
**시험 문제를 미리 보고 공부한 학생의 점수는 믿을 수 없습니다.** 모델도 마찬가지예요.
데이터의 70%로 공부(훈련)시키고, 모델이 **한 번도 본 적 없는 30%**로 시험(테스트)을 봅니다.
- `test_size=0.3` : 30%를 테스트용으로
- `random_state=42` : 섞는 방법을 고정 → 누가 실행해도 같은 결과
- `stratify=y` : 훈련/테스트에 품종 비율이 똑같이 들어가도록

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print('훈련 데이터:', X_train.shape[0], '병 / 테스트 데이터:', X_test.shape[0], '병')

## 3. 여러 모델 한꺼번에 학습하고 정확도 비교하기
2회차에 배운 세 가지 모델을 **같은 훈련 데이터로 학습**시키고 **같은 테스트 데이터로 시험**을 봅니다.
- `fit(X_train, y_train)` : 학습(공부)
- `score(X_test, y_test)` : 정확도(맞힌 비율)
- 결과를 `DataFrame`(표)로 모아 두면 비교가 쉽습니다. **어떤 모델이 훈련 점수는 높은데 테스트 점수는 낮은지** 눈여겨보세요.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

models = {
    'kNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    '결정트리 (깊이 제한 없음)': DecisionTreeClassifier(random_state=0),
    '로지스틱 회귀': LogisticRegression(max_iter=5000),
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)                                   # 학습
    rows.append({'모델': name,
                 '훈련 정확도': round(model.score(X_train, y_train), 3),
                 '테스트 정확도': round(model.score(X_test, y_test), 3)})
result = pd.DataFrame(rows)
result

> 💬 **읽는 법** — 결정트리는 훈련 정확도가 1.000(100%)인데 테스트는 그보다 낮습니다. 훈련 데이터를 **통째로 외운** 흔적이에요(과적합). kNN은 둘 다 낮은 편인데, 그 이유는 나중에(7단계) 밝혀집니다.

## 4. 혼동행렬 — 어디서 틀렸는지 들여다보기
정확도는 "몇 % 맞혔나"만 알려 줍니다. **어떤 품종을 어떤 품종으로 착각했는지**는 **혼동행렬**로 봅니다.
- 행 = 실제 품종, 열 = 모델이 예측한 품종. **대각선**이 정답, 대각선 밖이 틀린 것.
- `classification_report`는 품종별 **정밀도(precision)·재현율(recall)·F1**을 한 번에 보여 줍니다.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
model = models['결정트리 (깊이 제한 없음)']
y_pred = model.predict(X_test)                       # 테스트 데이터 예측
cm = confusion_matrix(y_test, y_pred)
print('혼동행렬 (행=실제, 열=예측):\n', cm)
ConfusionMatrixDisplay(cm, display_labels=['품종0', '품종1', '품종2']).plot(cmap='Blues')
plt.title('결정트리 혼동행렬 (테스트 데이터)'); plt.show()
print(classification_report(y_test, y_pred, target_names=['품종0', '품종1', '품종2']))

> 💬 **읽는 법** — `precision`(정밀도) = "그 품종이라고 말한 것 중 진짜 그 품종인 비율", `recall`(재현율) = "진짜 그 품종 중 찾아낸 비율", `f1-score` = 둘의 조화 평균. `support` = 테스트에 있던 실제 개수.

## 5. 정확도의 함정 — 불균형 데이터
**환자가 5%뿐인 검사 데이터**를 가정해 봅시다. "모두 정상"이라고만 답하는 엉터리 모델도 정확도는 95%입니다!
scikit-learn의 `DummyClassifier(strategy='most_frequent')`가 바로 그 엉터리 모델입니다. 정확도와 재현율을 함께 비교해 보세요.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score

rng = np.random.default_rng(0)
n = 1000
sick = rng.random(n) < 0.05                          # 5%만 환자(True)
temp = np.where(sick, rng.normal(38.5, 0.6, n), rng.normal(36.8, 0.4, n))   # 체온: 환자는 높음
Xh = temp.reshape(-1, 1); yh = sick.astype(int)
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, yh, test_size=0.3, random_state=0, stratify=yh)

dummy = DummyClassifier(strategy='most_frequent').fit(Xh_tr, yh_tr)   # "전부 정상" 모델
real  = LogisticRegression().fit(Xh_tr, yh_tr)                        # 진짜 학습한 모델
for name, m in [('전부 정상 모델', dummy), ('진짜 학습 모델', real)]:
    p = m.predict(Xh_te)
    print(f'{name:10s} 정확도 {accuracy_score(yh_te, p):.3f} | 재현율(환자 찾기) {recall_score(yh_te, p, zero_division=0):.3f} | 정밀도 {precision_score(yh_te, p, zero_division=0):.3f}')

> 💬 두 모델의 **정확도는 비슷**하지만 재현율은 완전히 다릅니다. 병 진단처럼 **놓치면 큰일 나는 문제**는 재현율을, 스팸 필터처럼 **헛경보가 싫은 문제**는 정밀도를 봐야 합니다.

## 6. 과적합을 눈으로 보기 — 결정트리 깊이 실험
결정트리의 `max_depth`(질문을 몇 단계까지 하는가)를 1부터 15까지 바꾸며 **훈련 정확도와 테스트 정확도**를 그려 봅니다.
- 깊이가 얕으면 둘 다 낮음(**과소적합**: 너무 단순)
- 깊이가 깊어지면 훈련은 100%가 되지만 테스트는 멈춤(**과적합**: 외워 버림)
- 우리가 원하는 곳은 **테스트 정확도가 가장 높은 깊이**입니다.

In [ ]:
depths = range(1, 16)
train_acc, test_acc = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    train_acc.append(t.score(X_train, y_train)); test_acc.append(t.score(X_test, y_test))

plt.figure(figsize=(7, 4))
plt.plot(depths, train_acc, 'o-', label='훈련 정확도'); plt.plot(depths, test_acc, 's-', label='테스트 정확도')
plt.xlabel('max_depth (트리 깊이)'); plt.ylabel('정확도'); plt.title('깊이에 따른 훈련/테스트 정확도'); plt.legend(); plt.grid(alpha=.3); plt.show()
best_depth = int(np.argmax(test_acc)) + 1
print('테스트 정확도가 가장 높은 깊이:', best_depth, '→ 정확도', round(max(test_acc), 3))

## 7. kNN이 약했던 이유 — 스케일링
kNN은 **거리**로 이웃을 찾습니다. 그런데 와인 성분 중 `proline`은 값이 수백~천 단위, `hue`는 1 근처입니다. 거리 계산이 큰 숫자 열에 **휘둘려** 버리죠.
`StandardScaler`로 모든 열을 **평균 0, 표준편차 1**로 맞추면 공정해집니다. `make_pipeline`은 "스케일링 → kNN"을 한 덩어리 모델로 묶어 줍니다.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
print('열마다 크기가 다르다 →\n', X.describe().loc[['mean', 'max'], ['alcohol', 'hue', 'proline']].round(2))

knn_raw    = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
knn_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(X_train, y_train)
print('\nkNN 스케일링 전 테스트 정확도:', round(knn_raw.score(X_test, y_test), 3))
print('kNN 스케일링 후 테스트 정확도:', round(knn_scaled.score(X_test, y_test), 3))

## 8. 교차검증 — 한 번 나누기의 "운"을 없애기
훈련/테스트를 딱 한 번 나누면 **어떤 데이터가 테스트에 들어갔느냐**에 따라 점수가 오르내립니다.
**5-fold 교차검증**은 훈련 데이터를 5조각으로 나누어 5번 시험을 보고 **평균**을 냅니다. `cross_val_score(모델, X, y, cv=5)` 한 줄이면 됩니다.
(테스트 데이터는 아직 쓰지 않습니다 — **마지막 확인용**으로 아껴 둡니다.)

In [ ]:
from sklearn.model_selection import cross_val_score
cands = {
    'kNN(k=5, 스케일링)': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    f'결정트리(깊이 {best_depth})': DecisionTreeClassifier(max_depth=best_depth, random_state=0),
    '로지스틱 회귀(스케일링)': make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
}
for name, m in cands.items():
    scores = cross_val_score(m, X_train, y_train, cv=5)
    print(f'{name:22s} 5번 점수 {np.round(scores, 3)} → 평균 {scores.mean():.3f} (±{scores.std():.3f})')

## 9. 하이퍼파라미터 튜닝 — GridSearchCV
`k`(kNN 이웃 수)나 `max_depth`(트리 깊이)처럼 **사람이 정해 주는 값**을 **하이퍼파라미터**라고 합니다.
`GridSearchCV`는 후보 값을 **전부 교차검증으로 시험**해 가장 좋은 값을 골라 줍니다.
- `param_grid` : 시험할 후보 목록. 파이프라인 안의 값은 `단계이름__옵션` 형식으로 씁니다.
- `best_params_` : 가장 좋았던 값, `best_score_` : 그때의 교차검증 평균 점수

In [ ]:
from sklearn.model_selection import GridSearchCV
# (1) kNN의 k 튜닝
grid_knn = GridSearchCV(make_pipeline(StandardScaler(), KNeighborsClassifier()),
                        param_grid={'kneighborsclassifier__n_neighbors': list(range(1, 26))}, cv=5)
grid_knn.fit(X_train, y_train)
print('kNN  최적 k =', grid_knn.best_params_['kneighborsclassifier__n_neighbors'], '| 교차검증 평균', round(grid_knn.best_score_, 3))

# (2) 결정트리의 깊이 튜닝
grid_tree = GridSearchCV(DecisionTreeClassifier(random_state=0),
                         param_grid={'max_depth': list(range(1, 11)), 'min_samples_leaf': [1, 2, 4]}, cv=5)
grid_tree.fit(X_train, y_train)
print('트리 최적 설정 =', grid_tree.best_params_, '| 교차검증 평균', round(grid_tree.best_score_, 3))

# k별 점수 그래프
ks = range(1, 26); ms = grid_knn.cv_results_['mean_test_score']
plt.figure(figsize=(7, 3.5)); plt.plot(list(ks), ms, 'o-'); plt.xlabel('k'); plt.ylabel('교차검증 정확도'); plt.title('kNN: k에 따른 점수'); plt.grid(alpha=.3); plt.show()

## 10. 개선 전 → 후 비교표 (테스트 데이터로 최종 확인)
이제 아껴 둔 **테스트 데이터**로 딱 한 번 최종 점수를 냅니다. 처음(3단계)의 기본 모델과 튜닝한 모델을 한 표에 놓고 비교하세요.
`best_estimator_`는 GridSearchCV가 찾은 최적 설정으로 **훈련 데이터 전체를 다시 학습한** 모델입니다.

In [ ]:
from sklearn.metrics import f1_score
final = {
    '[전] kNN (k=5, 스케일 X)': models['kNN (k=5)'],
    '[후] kNN (스케일 + k 튜닝)': grid_knn.best_estimator_,
    '[전] 결정트리 (깊이 제한 X)': models['결정트리 (깊이 제한 없음)'],
    '[후] 결정트리 (깊이·잎 튜닝)': grid_tree.best_estimator_,
    '[전] 로지스틱 회귀 (스케일 X)': models['로지스틱 회귀'],
    '[후] 로지스틱 회귀 (스케일 O)': cands['로지스틱 회귀(스케일링)'].fit(X_train, y_train),
}
rows = []
for name, m in final.items():
    p = m.predict(X_test)
    rows.append({'모델': name, '테스트 정확도': round(accuracy_score(y_test, p), 3),
                 'F1(평균)': round(f1_score(y_test, p, average='macro'), 3)})
compare = pd.DataFrame(rows)
compare

## 11. 회귀 모델의 평가 — MAE · RMSE · R²
분류가 아니라 **숫자를 맞히는 회귀**(2회차: 공부시간 → 성적)는 "맞다/틀리다"가 없으니 **얼마나 빗나갔는지**를 잽니다.
- **MAE** : 오차의 평균 크기(점) — "평균 6점쯤 틀린다"
- **RMSE** : 큰 오차를 더 무겁게 벌주는 평균 오차(점)
- **R²** : 1에 가까울수록 좋음, 0이면 "그냥 평균으로 찍은 것"과 같음
같은 데이터로 **선형회귀**와 **결정트리 회귀**를 비교합니다. (데이터는 2회차와 같은 규칙으로 만든 가상 데이터입니다.)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rng = np.random.default_rng(2026); n = 120
study = np.round(rng.uniform(0.5, 8, n), 1); sleep = np.round(rng.normal(6.5, 1, n), 1)
score = np.clip(40 + 6.5*study - 1.2*(sleep-7)**2 + rng.normal(0, 7, n), 0, 100).round(1)
df = pd.DataFrame({'공부시간': study, '수면시간': sleep, '성적': score})
Xr, yr = df[['공부시간', '수면시간']], df['성적']
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.3, random_state=1)

for name, m in [('선형회귀', LinearRegression()), ('결정트리 회귀(깊이 제한 X)', DecisionTreeRegressor(random_state=0)),
                ('결정트리 회귀(깊이 3)', DecisionTreeRegressor(max_depth=3, random_state=0))]:
    m.fit(Xr_tr, yr_tr); p = m.predict(Xr_te)
    print(f'{name:20s} MAE {mean_absolute_error(yr_te, p):5.2f}점 | RMSE {np.sqrt(mean_squared_error(yr_te, p)):5.2f}점 | R² {r2_score(yr_te, p):.3f}')

> 💬 결정트리 회귀도 깊이를 제한하지 않으면 **과적합**해서 오차가 커집니다. 분류든 회귀든 원리는 같아요.

## 🧩 직접 해보기 1 — 로지스틱 회귀의 C 값 튜닝
로지스틱 회귀에는 `C`라는 하이퍼파라미터가 있습니다(작을수록 단순한 모델). `GridSearchCV`로 `C` 후보 `[0.01, 0.1, 1, 10, 100]`을 시험해 최적값과 점수를 출력하세요.
힌트: 파이프라인 안의 로지스틱 회귀 옵션 이름은 `logisticregression__C` 입니다.

In [ ]:
grid_lr = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
                       param_grid={'logisticregression__C': [0.01, 0.1, 1, 10, 100]},   # TODO: 후보 목록이 맞는지 확인
                       cv=5)
grid_lr.fit(X_train, y_train)                        # TODO: 훈련 데이터로 fit
print('최적 C =', grid_lr.best_params_, '| 교차검증 평균', round(grid_lr.best_score_, 3))
print('테스트 정확도 =', round(grid_lr.score(X_test, y_test), 3))

<details><summary>▶ 정답 보기</summary>

```python
grid_lr = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
                       param_grid={'logisticregression__C': [0.01, 0.1, 1, 10, 100]}, cv=5)
grid_lr.fit(X_train, y_train)
print('최적 C =', grid_lr.best_params_, '| 교차검증 평균', round(grid_lr.best_score_, 3))
print('테스트 정확도 =', round(grid_lr.score(X_test, y_test), 3))
```
</details>

## 🧩 직접 해보기 2 — 재현율이 중요한 상황 만들기
5단계의 체온 데이터에서, 진짜 학습 모델(`real`)이 **환자라고 판정하는 기준(확률 문턱값)**을 0.5에서 0.2로 낮추면 재현율과 정밀도가 어떻게 바뀔까요?
`predict_proba(...)[:, 1]`은 "환자일 확률"입니다. 문턱값을 바꿔 보고 표를 완성하세요.

In [ ]:
proba = real.predict_proba(Xh_te)[:, 1]              # 환자일 확률
for thr in [0.5, 0.3, 0.2, 0.1]:                     # TODO: 다른 문턱값도 넣어 보세요
    p = (proba >= thr).astype(int)                   # 확률이 문턱값 이상이면 환자(1)
    print(f'문턱값 {thr:.1f} → 재현율 {recall_score(yh_te, p):.3f} | 정밀도 {precision_score(yh_te, p, zero_division=0):.3f} | 정확도 {accuracy_score(yh_te, p):.3f}')

<details><summary>▶ 정답(해설) 보기</summary>

문턱값을 낮추면 "조금만 의심돼도 환자"라고 판정하므로 **재현율은 올라가고 정밀도는 내려갑니다**. 병 진단이라면 재현율을 택하고, 대신 헛경보는 2차 검사로 거르는 식으로 설계합니다. 두 지표는 **시소** 관계입니다.
</details>

## 📝 오늘의 결과물 제출 — "개선 방향 결정 근거"
아래 셀에 이름·학번을 쓰고, 표를 보며 **어떤 모델을 최종 선택할지, 왜 그런지** 2~3문장을 적은 뒤 실행하세요. 출력 화면을 캡처해 제출합니다.

In [ ]:
이름 = '홍길동'          # TODO: 본인 이름
학번 = '10101'          # TODO: 본인 학번
최종_선택_모델 = '[후] 로지스틱 회귀 (스케일 O)'     # TODO: compare 표의 모델 이름 중 하나
선택_근거 = """
1. 스케일링한 로지스틱 회귀가 테스트 정확도와 F1이 가장 높았다.
2. 8단계 교차검증에서도 평균이 가장 높고 편차(±)가 가장 작아, 한 번의 운이 아니라고 판단했다.
3. kNN은 스케일링+k 튜닝으로 가장 많이 개선됐지만 최종 점수는 로지스틱 회귀가 더 높았다. 다음에는 품종별 재현율을 확인해 약한 품종을 보완하겠다.
"""                     # TODO: 본인의 근거로 바꾸기

print('=' * 60); print(f'3회차 실습 결과물  |  {이름} ({학번})'); print('=' * 60)
print(compare.to_string(index=False))
print('-' * 60); print('최종 선택:', 최종_선택_모델); print('근거:', 선택_근거)

## ✅ 자기 점검 체크리스트
- [ ] 훈련/테스트를 나누는 이유를 "시험 문제 미리 보기"에 빗대어 설명할 수 있다.
- [ ] 혼동행렬에서 대각선이 무엇인지, 정밀도와 재현율이 어떻게 다른지 말할 수 있다.
- [ ] 정확도 95%가 좋은 모델이 **아닐 수도 있는** 예를 들 수 있다.
- [ ] 트리 깊이 그래프에서 과소적합·과적합 구간을 가리킬 수 있다.
- [ ] 교차검증과 GridSearchCV가 각각 무엇을 해 주는지 말할 수 있다.
- [ ] 개선 전후 표를 근거로 최종 모델을 선택하고 이유를 썼다.

**다음 시간(4회차)**: 정답이 없는 데이터를 스스로 묶는 **비지도학습**(군집·차원축소)